# CIVICLEAR NLP Training Notebook

This notebook trains the text/NLP model for CIVICLEAR/DILG-RC.

Input: citizen report description  
Output: violation type + confidence

In [ ]:
!pip install -q pandas scikit-learn joblib matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Set project path

Upload the `civiclear-nlp-training` folder to your Google Drive, then check that this path is correct.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/civiclear-nlp-training')
CSV_PATH = PROJECT_DIR / 'data' / 'civiclear_nlp_text_dataset_500.csv'
OUTPUT_DIR = PROJECT_DIR

print('Project folder:', PROJECT_DIR)
print('CSV path:', CSV_PATH)
print('CSV exists:', CSV_PATH.exists())

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(df.head())
print('Rows:', len(df))
print(df['label'].value_counts())

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib, json

# Clean data
df = df[['text', 'label']].copy()
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip()
df = df.dropna()
df = df[(df['text'] != '') & (df['label'] != '')]

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))

In [ ]:
nlp_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=10000
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='liblinear'
    ))
])

nlp_model.fit(train_df['text'], train_df['label'])
print('NLP training complete.')

In [ ]:
y_true = test_df['label']
y_pred = nlp_model.predict(test_df['text'])

accuracy = accuracy_score(y_true, y_pred)
print('Accuracy:', round(accuracy * 100, 2), '%')
print(classification_report(y_true, y_pred, zero_division=0))

In [ ]:
# Save model and reports
model_dir = PROJECT_DIR / 'models'
report_dir = PROJECT_DIR / 'reports'
model_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / 'civiclear_nlp_model.joblib'
joblib.dump(nlp_model, model_path)

report_text = classification_report(y_true, y_pred, zero_division=0)
(report_dir / 'classification_report.txt').write_text(
    f'Accuracy: {accuracy * 100:.2f}%\n\n{report_text}',
    encoding='utf-8'
)

report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
with open(report_dir / 'nlp_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'accuracy': accuracy, 'classification_report': report_dict}, f, indent=4)

print('Saved model:', model_path)

In [ ]:
labels = sorted(df['label'].unique())
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=range(len(labels)),
    yticks=range(len(labels)),
    xticklabels=labels,
    yticklabels=labels,
    ylabel='True Label',
    xlabel='Predicted Label',
    title='CIVICLEAR NLP Confusion Matrix'
)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
fig.tight_layout()
cm_path = report_dir / 'confusion_matrix.png'
fig.savefig(cm_path, dpi=200, bbox_inches='tight')
print('Saved confusion matrix:', cm_path)

In [ ]:
sample_texts = [
    'May sasakyan na nakaparada sa kalsada',
    'May basura na nakaharang sa sidewalk',
    'May hollow blocks at buhangin sa daan',
    'May vendor na nakaharang sa bangketa',
    'Malinis ang daan at walang obstruction'
]

predictions = nlp_model.predict(sample_texts)
probabilities = nlp_model.predict_proba(sample_texts)

rows = []
for text, pred, prob in zip(sample_texts, predictions, probabilities):
    confidence = max(prob) * 100
    rows.append({'text': text, 'prediction': pred, 'confidence': round(confidence, 2)})
    print('Text:', text)
    print('Prediction:', pred)
    print('Confidence:', round(confidence, 2), '%')
    print('-' * 60)

pd.DataFrame(rows).to_csv(report_dir / 'sample_predictions.csv', index=False)
print('Saved sample predictions:', report_dir / 'sample_predictions.csv')